### Structured Output

Structured output allows an LLM to return responses in a predefined format, such as JSON or a Python object, making the output easier to process programmatically.

LangChain provides built-in support for generating and handling structured outputs from LLMs.

### Pydantic

Pydantic is a Python library used for data validation and defining structured data using Python classes.

It is commonly used with LangChain to define the expected structure of an LLM's output.

In [4]:
import os
from langchain.chat_models import init_chat_model

os.environ["GROQ_API_KEY"] = os.getenv("GROQ_API_KEY")

model = init_chat_model(
    model="openai/gpt-oss-20b",
    model_provider="groq"
)

response = model.invoke("Explain machine learning in one sentence.")

print(response.content)

Machine learning is a branch of artificial intelligence that trains algorithms on data so they can automatically discover patterns and make predictions or decisions without being explicitly programmed for each task.


In [5]:
model

ChatGroq(metadata={'lc_versions': {'langchain-core': '1.5.3', 'langchain': '1.3.14'}}, output_version=None, profile={'name': 'GPT OSS 20B', 'release_date': '2025-08-05', 'last_updated': '2026-05-27', 'open_weights': True, 'max_input_tokens': 131072, 'max_output_tokens': 65536, 'text_inputs': True, 'image_inputs': False, 'audio_inputs': False, 'video_inputs': False, 'text_outputs': True, 'image_outputs': False, 'audio_outputs': False, 'video_outputs': False, 'reasoning_output': True, 'tool_calling': True, 'structured_output': True, 'attachment': False, 'temperature': True}, client=<groq.resources.chat.completions.Completions object at 0x7704634c8560>, async_client=<groq.resources.chat.completions.AsyncCompletions object at 0x7704627faea0>, model_name='openai/gpt-oss-20b', model_kwargs={}, groq_api_key=SecretStr('**********'), groq_api_base=None, groq_proxy=None)

In [6]:
from pydantic import BaseModel, Field

# BaseModel is used to define the structure of the output
class Movie(BaseModel):

    # Field descriptions help the LLM understand each expected value
    title: str = Field(description="Title of the movie")

    year: int = Field(description="Year the movie was released")

    director: str = Field(description="Director of the movie")

    rating: float = Field(description="Movie rating out of 10")

### Pydantic Model

A Pydantic model defines a structured schema for the expected output.

- **BaseModel** – Creates a data model with validation.
- **Field** – Adds descriptions and additional information for each field.
- **Data Types** – Define the expected type of each output value.

This schema can be used with LangChain to make an LLM return structured and validated data.

In [ ]:
model_with_stucture=model.with_structured_output(Movie)

In [10]:
model_with_stucture

_ChatModelBinding(bound=ChatGroq(metadata={'lc_versions': {'langchain-core': '1.5.3', 'langchain': '1.3.14'}}, output_version=None, profile={'name': 'GPT OSS 20B', 'release_date': '2025-08-05', 'last_updated': '2026-05-27', 'open_weights': True, 'max_input_tokens': 131072, 'max_output_tokens': 65536, 'text_inputs': True, 'image_inputs': False, 'audio_inputs': False, 'video_inputs': False, 'text_outputs': True, 'image_outputs': False, 'audio_outputs': False, 'video_outputs': False, 'reasoning_output': True, 'tool_calling': True, 'structured_output': True, 'attachment': False, 'temperature': True}, client=<groq.resources.chat.completions.Completions object at 0x7704634c8560>, async_client=<groq.resources.chat.completions.AsyncCompletions object at 0x7704627faea0>, model_name='openai/gpt-oss-20b', model_kwargs={}, groq_api_key=SecretStr('**********'), groq_api_base=None, groq_proxy=None), kwargs={'tools': [{'type': 'function', 'function': {'name': 'Movie', 'description': '', 'parameters':

In [11]:
model_with_stucture.invoke("provide the detailds of the movie your name")

Movie(title='Your Name', year=2016, director='Makoto Shinkai', rating=9.0)

### message output alogside with stucture

In [13]:
model_with_stucture=model.with_structured_output(Movie,include_raw=True)
response=model_with_stucture.invoke("provide the detailds of the inseption")

In [16]:
response # The response contains both the raw AI message and the parsed structured output

{'raw': AIMessage(content='', additional_kwargs={'reasoning_content': 'The user: "provide the detailds of the inseption". They likely mean "Inception" the movie. They want details. Use function? They want details of the movie "Inception". Use function "Movie" with director, rating, title, year. Probably we should call the function. Provide details. Use the function to get details. Let\'s call it.', 'tool_calls': [{'id': 'fc_f0d87ff7-2ed3-4eab-b36d-c26755071b96', 'function': {'arguments': '{"director":"Christopher Nolan","rating":8.8,"title":"Inception","year":2010}', 'name': 'Movie'}, 'type': 'function'}]}, response_metadata={'token_usage': {'completion_tokens': 115, 'prompt_tokens': 155, 'total_tokens': 270, 'completion_time': 0.120732274, 'completion_tokens_details': {'reasoning_tokens': 76}, 'prompt_time': 0.007655761, 'prompt_tokens_details': None, 'queue_time': 0.364847862, 'total_time': 0.128388035}, 'model_name': 'openai/gpt-oss-20b', 'system_fingerprint': 'fp_5979a0e1b7', 'serv

### Nested Structure

In [17]:
from pydantic import BaseModel, Field


# Defines the structure for each actor
class Actor(BaseModel):
    name: str
    role: str


# Defines the complete movie structure
class MovieDetails(BaseModel):
    title: str
    year: int

    # List of Actor objects (nested structure)
    cast: list[Actor]

    # List of movie genres
    genre: list[str]

    #Optional budget field
    budget: float | None = Field(
        default=None,
        description="Budget in Million USD"
    )

In [19]:
model_with_stucture = model.with_structured_output(MovieDetails)
response = model_with_stucture.invoke("provide the detials of the Movie inception")

In [20]:
response

MovieDetails(title='Inception', year=2010, cast=[Actor(name='Leonardo DiCaprio', role='Dom Cobb'), Actor(name='Joseph Gordon-Levitt', role='Arthur'), Actor(name='Elliot Page', role='Ariadne'), Actor(name='Tom Hardy', role='Eames'), Actor(name='Ken Watanabe', role='Saito'), Actor(name='Cillian Murphy', role='Robert Fischer')], genre=['Action', 'Science Fiction', 'Thriller'], budget=160000000.0)